# 5차시 강사용 Notebook — 데이터 훑어보기와 정리(EDA)

**시연 전 안내**: Orange3로 File → Data Table → Feature Statistics를 먼저 보여준 뒤 이 노트북으로 넘어옵니다.

## 1단계. 데이터 읽고 결측값 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week05/week05_dirty_process_data.csv")
print(df.shape)
print(df.isna().sum())

(113, 9)
측정시간        0
로트번호        0
설비번호        0
공정명         0
온도_섭씨       3
압력_Pa       2
가스유량_slm    3
처리시간_sec    0
합격여부        2
dtype: int64


## 2단계. 결측값 처리하기
**설명 포인트**: 범주형(합격여부)은 제거, 숫자형은 중앙값 대체라는 원칙을 칠판에 적어둔다.

In [2]:
df = df.dropna(subset=["합격여부"])

for col in ["온도_섭씨", "압력_Pa", "가스유량_slm"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print(df.isna().sum().sum())

0


## 3단계. 중복값 찾고 제거하기

In [3]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

3
(108, 9)


## 4단계. 이상값 찾기
**어려워할 부분**: isna()로 이상값이 안 잡힌다는 것을 직접 실행해서 보여준다.

In [4]:
hot_outlier = df[df["온도_섭씨"] > 400]
neg_pressure = df[df["압력_Pa"] < 0]
bad_gas = df[(df["가스유량_slm"] < 0) | (df["가스유량_slm"] >= 200)]

print(len(hot_outlier), len(neg_pressure), len(bad_gas))

2 2 2


## 5단계. 이상값 제거하기

In [5]:
df = df[df["온도_섭씨"] <= 400]
df = df[df["압력_Pa"] >= 0]
df = df[(df["가스유량_slm"] >= 0) & (df["가스유량_slm"] < 200)]

print(df.shape)

(102, 9)


## 6단계(종합). 정제 전후 평균 비교하기

In [6]:
after_mean = df["온도_섭씨"].mean()
print(f"정제 전 평균 온도: 304.9도 (113행)")
print(f"정제 후 평균 온도: {after_mean:.1f}도 ({len(df)}행)")

정제 전 평균 온도: 304.9도 (113행)
정제 후 평균 온도: 300.6도 (102행)


## 7단계. 오류 대처 방법
- `KeyError` 발생 시: `df.columns`로 정확한 열 이름 확인.
- fillna 대상 지정 실수: 반드시 `df["열"] = df["열"].fillna(...)` 형태인지 확인.

## 확장 실습(빠른 학습자용)
공정명 오타를 str.strip()/replace()로 통일해보게 한다.

In [7]:
df["공정명"] = df["공정명"].str.strip().replace({"중착": "증착", "포토공정": "포토"})
print(df["공정명"].unique())

['포토' '산화' '세정' '증착' '식각']
